# world-platform · demo
**AINative Blank IDE + World Middleware Runtime**

---

| 模块 / Module | 说明 / Description |
|--------------|-------------------|
| `WorldKernel` | 统一内核 / Unified kernel — Schema + Trace + Memory + Planner |
| `%%world / %%dev` | 自然语言驱动世界状态 / Natural language drives world state |
| `Pipeline` | 链式多步 LLM 推理 / Chainable multi-step LLM reasoning |
| 🧬 Self-Evolution | IDE 自主生成新命令 / IDE autonomously creates new commands |

---
修改下方 `API_KEY` 后顺序运行所有单元。  
Edit `API_KEY` in the config cell below, then run all cells in order.

In [ ]:
# ============================================================
# 全局配置 — 运行前修改 API_KEY / Global config — edit API_KEY before running
# ============================================================
import os

API_KEY  = "sk-"  # ← 替换为你的 key / Replace with your key
MODEL    = "deepseek-v4-flash"                     # 模型名 / Model name
BASE_URL = "https://api.deepseek.com"              # 本地模型改这里 / Change for local models

# 同步到环境变量，供魔法命令读取 / Sync to env vars for magic commands
os.environ["DEEPSEEK_API_KEY"]  = API_KEY
os.environ["DEEPSEEK_MODEL"]    = MODEL
os.environ["DEEPSEEK_BASE_URL"] = BASE_URL

print(f"✓ Model : {MODEL}")
print(f"✓ Key   : {API_KEY[:8]}{'*' * (len(API_KEY)-8)}")

✓ Model : deepseek-v4-flash
✓ Key   : sk-ecf36***************************


In [49]:
# 热重载所有 runtime 模块并启动 Blank IDE（含 Observer 自动监听）
# Hot-reload all runtime modules and start Blank IDE (Observer auto-starts)
import importlib, runtime.observer, runtime.evolver, runtime.magic
importlib.reload(runtime.observer)
importlib.reload(runtime.evolver)
importlib.reload(runtime.magic)
%reload_ext runtime.magic

✓ Blank IDE 已加载 / Blank IDE loaded
  %ide_evolve    — 触发 IDE 进化    / Trigger IDE evolution
  %ide_history   — 查看进化历史     / View evolution history
  %ide_observe   — 查看执行观察日志 / View observation log
  %%world        — 自然语言写世界状态 / Write world state in natural language
  %%dev [--exec|--plan|--stream]  — AI 助手 / AI assistant


---
## 1 · WorldKernel — 统一内核 / Unified Runtime Kernel

WorldKernel 把所有 runtime 组件聚合为一个入口对象，并与 `%%world` 魔法命令共享同一份世界状态。  
WorldKernel aggregates all runtime components into a single object, sharing world state with `%%world` magic commands.

| 方法 / Method | 说明 / Description |
|--------------|-------------------|
| `k.set(key, value)` | 写入世界状态字段 / Write world state field |
| `k.snapshot(name)` | 保存当前状态快照 / Save state snapshot |
| `k.restore(name)` | 从快照回滚 / Roll back from snapshot |
| `k.plan(goal)` | LLM 自动规划步骤 / LLM auto-plan steps |
| `k.state()` | 查看状态摘要 / View state summary |

**关键 / Key**: `sync_magic=True`（默认）——代码里 `k.set(...)` 与 `%%world` 操作同一份 Schema。  
**Key**: `sync_magic=True` (default) — `k.set(...)` in code and `%%world` in magic operate on the **same** Schema.

In [ ]:
from kernel import WorldKernel

# 创建内核，sync_magic=True 将 Schema 注入 magic 全局状态
# Create kernel — sync_magic=True injects Schema into magic global state
k = WorldKernel(api_key=API_KEY, model=MODEL, base_url=BASE_URL)

# 写入项目基础字段 / Write project baseline fields
k.set("project",  "world-platform")
k.set("phase",    "prototype")
k.set("owner",    "Alice")
k.set("priority", "high")

# 保存快照 / Save snapshot
k.snapshot("baseline")
print("✓ Snapshot 'baseline' saved")
print(k)

# 模拟实验性变更，然后一键回滚 / Simulate experimental change, then roll back
k.set("phase", "experimental-broken")
print(f"\n⚠  phase changed → {k.get('phase')}")

k.restore("baseline")
print(f"✓  restored    → phase = {k.get('phase')}")

# 查看当前世界状态摘要 / Print current world state summary
print("\n" + k.state())

---
## 2 · Blank IDE Magic — 自然语言驱动世界状态 / Natural Language Drives World State

在 Jupyter 里用自然语言操作世界状态，LLM 负责解析与执行。  
Operate world state with natural language inside Jupyter — LLM handles parsing and execution.

| 命令 / Command | 说明 / Description |
|---------------|-------------------|
| `%%world` | 自然语言 → Schema<br>Natural language → writes to Schema |
| `%world_state` | 打印当前状态 / Print current state |
| `%%dev --exec` | 白板模式：LLM 决定并执行 IDE 动作<br>Whiteboard: LLM decides and executes IDE actions |
| `%%dev --plan` | 自动规划步骤列表 / Auto-plan step list |
| `%%dev --stream` | 流式对话 / Streaming chat |

In [ ]:
%%world
Project is world-platform, an AI Runtime framework. Current phase is prototype. Missing module: kernel_scheduler. Owner: Alice. Priority: high.

✅ **世界状态已更新**

## World State
- project (generic): world-platform AI Runtime
- phase (generic): prototype
- missing_component (generic): kernel scheduler
- owner (generic): Alice
- priority (generic): high

In [ ]:
# 查看 magic 写入后的世界状态（与 WorldKernel 的 Schema 同步）
# View world state after magic write (synced with WorldKernel's Schema)
%world_state

In [ ]:
%%dev --exec
Update phase to in-development and add missing_module field with value kernel_scheduler

## Kernel Scheduler Core Interface

Based on the project context (world-platform AI Runtime, prototype phase, missing kernel scheduler), the core interface should provide:

- **Kernel submission** with priority hints and resource requirements.
- **Dynamic scheduling** over heterogeneous devices (CPU, GPU, NPU, etc.).
- **Cancelation, pre‑emption** and **status querying**.
- **Resource accounting** (memory, compute units).
- **Asynchronous completion callbacks**.

### Data Structures

```c
// Kernel descriptor – what to execute
typedef struct {
    char* name;                 // e.g. "matmul_fp16"
    void* bin;                  // compiled binary / bytecode
    size_t bin_size;
    uint32_t grid_dim[3];       // for GPU-style launch
    uint32_t block_dim[3];
    void* args;                 // argument list
    size_t args_size;
    uint32_t device_type;       // CPU=0, GPU=1, NPU=2, …
} kernel_desc_t;

// Device identifier
typedef struct {
    uint32_t device_type;
    uint32_t device_index;      // 0,1,2...
} device_id_t;

// Resource request
typedef struct {
    size_t memory_bytes;        // required per device
    uint32_t compute_units;     // e.g. SM count, thread count
    uint32_t priority;          // 0 (low) … 255 (high)
    uint32_t timeout_ms;        // max execution time
} resource_req_t;

// Kernel handle returned after submission
typedef struct {
    uint64_t id;
    device_id_t assigned_device;
    volatile enum { QUEUED, RUNNING, DONE, FAILED, CANCELLED } status;
    size_t submit_time_ns;
    size_t start_time_ns;
    size_t end_time_ns;
} kernel_handle_t;
```

### Core API Functions

```c
// Submit a kernel for execution
kernel_handle_t* scheduler_submit(
    kernel_desc_t* desc,
    resource_req_t* req,
    void (*on_complete)(kernel_handle_t*),
    void* user_data
);

// Cancel a submitted kernel (best-effort)
int scheduler_cancel(kernel_handle_t* handle);

// Query current status (non-blocking)
kernel_status_t scheduler_get_status(kernel_handle_t* handle);

// Wait until kernel finishes (blocking)
int scheduler_wait(kernel_handle_t* handle);

// Update priority dynamically (if not yet started)
int scheduler_set_priority(kernel_handle_t* handle, uint32_t new_priority);

// Get current load / device availability
typedef struct {
    device_id_t dev;
    float utilization;          // 0.0 .. 1.0
    size_t free_memory;
    size_t queue_depth;
} device_stats_t;

device_stats_t* scheduler_get_device_stats(device_id_t dev);

// Shutdown the scheduler (cleanup)
void scheduler_shutdown();
```

### Usage Example (Prototype)

```c
void matmul_done(kernel_handle_t* h) {
    printf("Kernel %lu finished on device %u in %lu us\n",
           h->id, h->assigned_device.device_index,
           (h->end_time_ns - h->start_time_ns) / 1000);
}

kernel_desc_t desc = { .name = "matmul_fp16", /* ... */ };
resource_req_t req = { .memory_bytes = 256*1024*1024, 
                        .compute_units = 1, 
                        .priority = 200 };

kernel_handle_t* h = scheduler_submit(&desc, &req, matmul_done, NULL);
// ... do other work ...
scheduler_wait(h);   // or rely on callback
```

### Design Rationale (for prototype phase)

- **Simplicity** – explicit device assignment is done internally; the API hides scheduling policy.
- **Priority & pre‑emption** – high‑priority kernels can pre‑empt low‑priority ones (if runtime/device supports it). This is essential for mixed AI workloads (e.g., inference vs training).
- **Callbacks** – enable non‑blocking workflows common in AI runtimes.
- **Resource req** – allows the scheduler to gang‑schedule or partition devices.
- **Timeout** – avoid hanging on mis‑compiled kernels.

This interface should be implemented by **Alice** as a C‑based module (or C++/Rust for safety) before moving to later phases. The prototype should focus on correctness and basic fairness; advanced features (load balancing, migration) can be added later.

**Priority level:** High – blocking core runtime function.

---
## 3 · Pipeline — 链式多步 LLM 推理 / Chainable Multi-Step LLM Reasoning

声明式组合多步 LLM 推理，每步输出自动成为下步的 `{{input}}`。  
Declaratively compose multi-step LLM reasoning — each step's output automatically becomes the next step's `{{input}}`.

```python
result = (
    Pipeline(api_key=..., model=...)
    .step("Extract requirements from: {{input}}")
    .step("Convert to numbered task list: {{input}}")
    .step("Estimate effort S/M/L for each task: {{input}}")
    .run_final("Build a user auth system")
)
```

In [ ]:
from kernel import Pipeline

# 三步链式推理：提取挑战 → 提出方案 → 排优先级
# Three-step chain: extract challenges → propose solutions → prioritize
pipe = (
    Pipeline(api_key=API_KEY, model=MODEL, base_url=BASE_URL)
    # 步骤 1：提取核心技术挑战 / Step 1: extract core technical challenges
    .step(
        "Extract the top 3 technical challenges from this project description. "
        "Be concise, one line each.\n\n{{input}}"
    )
    # 步骤 2：为每个挑战提出解决方案 / Step 2: propose solution for each challenge
    .step(
        "For each challenge, propose a concrete solution approach in 1-2 sentences:\n\n{{input}}"
    )
    # 步骤 3：按优先级标注 / Step 3: label by priority
    .step(
        "Label each solution as P0 / P1 / P2 based on impact and urgency. "
        "Give a one-line reason for each label:\n\n{{input}}"
    )
)

# 项目简介作为输入 / Project brief as pipeline input
project_brief = (
    "world-platform: AINative IDE runtime framework.\n"
    "Goals: kernel scheduler, world state management, "
    "LLM integration, self-evolving IDE commands.\n"
    "Current phase: prototype. Team size: 1 engineer."
)

# 运行流水线，仅返回最终结果 / Run pipeline, return only final output
result = pipe.run_final(project_brief)
print(result)

---
## 4 · 🧬 Self-Evolving IDE — IDE 自主进化 / IDE Self-Evolution

IDE 观察你的使用行为，以架构师身份设计并注入你没意识到自己需要的新功能。  
The IDE observes your behavior and, acting as an architect, designs and injects capabilities you didn't realize you needed.

**流程 / Flow**

    Observer 静默记录执行 → %ide_evolve → LLM 作为架构师分析使用模式
    → 提出新功能 → 热注入内核 → 新命令立刻可用

    Observer silently records → %ide_evolve → LLM as architect analyzes usage patterns
    → Proposes new capability → hot-injects into kernel → new command immediately available

| 命令 / Command | 说明 / Description |
|---------------|-------------------|
| `%ide_observe` | 查看执行观察日志 / View execution observation log |
| `%ide_evolve` | 触发一次 IDE 进化 / Trigger one evolution cycle |
| `%ide_history` | 查看所有进化历史 / View all evolution history |
| `%%ide` | 用自然语言描述新功能，立即注入<br>Describe new capability in NL, inject immediately |

In [ ]:
# 查看 Observer 静默记录到的执行日志 / View execution log recorded silently by Observer
%ide_observe

## 👁 执行观察日志 / Execution Observation Log
- 总执行次数 / Total executions : **2**
- 错误次数   / Error count      : **0**

**最近记录 / Recent records**:
```
✓  # 热重载所有 runtime 模块并启动 Blank IDE（含 Observer 自动监听） ↵ # Hot-reload all runtime modules and start Blank IDE (with Observer aut
✓  %%world ↵ world-platform AI Runtime 框架，当前阶段 prototype，缺少 kernel 调度器，负责人 Alice，优先级 high
```

In [50]:
# 触发一次 IDE 自进化 / Trigger one IDE self-evolution cycle
# LLM 以架构师身份分析使用行为 → 提出新功能 → 热注入内核
# LLM acts as architect, analyzes behavior → proposes new capability → hot-injects into kernel
%ide_evolve

🧬 IDE 进化引擎启动 / Evolution engine thinking...

═══════════════════════════════════════════════════════
🌱 进化完成 / Evolution complete
   新能力 / New capability : A two-in-one magic command that records timestamped developer thoughts, variable snapshots, and last output (cell mode), and retrieves/clears journal entries (line mode).
   新命令 / New command    : %journal 或/or %%journal
   分析   / Analysis       : Developers frequently make code changes without documenting their rationale, leading to lost context when revisiting cod...
═══════════════════════════════════════════════════════


In [52]:
# 查看 IDE 进化历史，IDE 获得了哪些新能力 / View evolution history — what new capabilities did IDE gain?
%ide_history

## 🧬 自主进化 / Autonomous Evolutions (1 次/times)

### 1. ✅ `%journal`
**功能 / Capability**: A two-in-one magic command that records timestamped developer thoughts, variable snapshots, and last output (cell mode), and retrieves/clears journal entries (line mode).

**分析 / Analysis**: Developers frequently make code changes without documenting their rationale, leading to lost context when revisiting code later. While version control captures what changed, it rarely captures why. Th...


---
## 5 · 🌱 新生能力展示 — IDE 自主创造的 `%%journal` / Newly Born Capability — IDE-Created `%%journal`

这个命令**不是人写的**，是上一步 IDE 自主进化时由 LLM 作为架构师设计并热注入的。  
This command was **not hand-written** — it was designed by LLM as an architect during the previous evolution step and hot-injected into the kernel.

| 用法 / Usage | 说明 / Description |
|-------------|-------------------|
| `%%journal` + 任意文字 | 记录带时间戳的开发思路，同时快照当前变量<br>Record a timestamped dev note, snapshot current variables |
| `_journal_entries` | 直接访问所有日志条目 / Access all entries directly |

> IDE 观察到：开发者频繁改动代码却不记录原因 → 自动设计了 `%%journal` 补上这个缺口。  
> IDE observed: developers change code frequently without documenting rationale → autonomously designed `%%journal` to fill this gap.

In [53]:
%%journal
Decided to use Pipeline for multi-step reasoning instead of chaining LLMCell manually.
Reason: Pipeline gives cleaner syntax and the {{input}} template makes data flow explicit.

In [54]:
%%journal
Snapshot after WorldKernel baseline — project=world-platform, phase=prototype, owner=Alice.

In [62]:
from IPython.display import display, Markdown

# 直接读取 IDE 自动创建的 journal 存储变量
# Directly read the journal storage variable auto-created by IDE
entries = [e for e in _journal_entries if e.get("note", "").strip() not in ("show", "")]

lines = [f"## 📓 Journal — {len(entries)} 条开发笔记 / Developer Notes\n"]
for i, e in enumerate(entries, 1):
    lines.append(
        f"**#{i}** `{e['time']}`  \n"
        f"> {e['note']}\n"
    )
display(Markdown("\n".join(lines)))

## 📓 Journal — 2 条开发笔记 / Developer Notes

**#1** `2026-05-13 00:21:57`  
> Decided to use Pipeline for multi-step reasoning instead of chaining LLMCell manually.
Reason: Pipeline gives cleaner syntax and the {{input}} template makes data flow explicit.

**#2** `2026-05-13 00:22:01`  
> Snapshot after WorldKernel baseline — project=world-platform, phase=prototype, owner=Alice.
